In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00


In [3]:
# ===============================
# INSTALL (RUN ONCE)
# ===============================
# !pip install transformers sentence-transformers torch scikit-learn pandas openpyxl -q

# ===============================
# IMPORTS
# ===============================
import pandas as pd
import numpy as np
import torch
import re
import json
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, classification_report

# ===============================
# LOAD MODELS
# ===============================
print("Loading embedding model...")
embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("Loading FLAN-T5-Large...")
model_name = "google/flan-t5-large"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
llm_model  = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Models ready.\n")

# ===============================
# FIX 1 — ALL 6 CATEGORIES WITH DEFINITIONS
# Previously only had 4 — Pet Food and Other were missing
# ===============================
category_defs = {
    "Food": """Food is composed of all edible goods purchased and consumed by
    the household for nourishment. Includes cereals, meat, fish, seafood,
    dairy products, eggs, oils, fats, fruits, nuts, vegetables, tubers,
    sugar, confectionery, desserts, salt, sauces, condiments, spices,
    herbs, bread, pastry, frozen food, canned food, pasta, rice, snacks,
    chocolate, jams, honey, baby food and cooking ingredients.""",

    "Drinks": """All kinds of beverages purchased by the household.
    Includes water, mineral water, soft drinks, juices, energy drinks,
    sports drinks, tea, coffee, milk drinks, alcoholic beverages including
    beer, wine, spirits, whisky, champagne and cider.""",

    "Home Care": """Household cleaning and maintenance products.
    Includes detergents, dishwashing liquid, laundry products, fabric
    conditioner, bleach, surface cleaners, bathroom cleaners, window
    cleaners, insecticides, air fresheners, bin bags, kitchen foil,
    cling film, cleaning accessories and other household care items.""",

    "Personal Care": """Products for personal hygiene, grooming and
    beautification. Includes shampoo, shower gel, soap, toothpaste,
    deodorants, skincare, cosmetics, hair care, oral hygiene, toilet
    tissue, tissues, napkins, nappies, vitamins and personal health
    products.""",

    # FIX: Added Pet Food
    "Pet Food": """Products specifically manufactured for animal
    consumption. Includes dog food, cat food, pet treats, pet nutrition
    products and animal feed.""",

    # FIX: Added Other
    "Other": """Miscellaneous products that do not fit into Food, Drinks,
    Home Care or Personal Care. Includes batteries, lightbulbs, shoe care
    products, stationery and other non-food non-care items.""",
}

categories = list(category_defs.keys())
print(f"Categories: {categories}\n")

# ===============================
# FIX 2 — CORRECT LABEL MAPPING
# Maps Excel Super_Category values to clean labels exactly
# ===============================
LABEL_MAP = {
    "DRINKS":        "Drinks",
    "FOODS":         "Food",
    "HOME CARE":     "Home Care",
    "PERSONAL CARE": "Personal Care",
    "PET FOOD":      "Pet Food",
    "OTHER":         "Other",
}

def clean_label(x):
    return LABEL_MAP.get(str(x).strip().upper(), "Other")

# ===============================
# LOAD DATA
# ===============================
df = pd.read_excel("categories_with_descriptions_and_labels.xlsx")
df.columns = df.columns.str.strip()
df["GT"]   = df["Super_Category"].apply(clean_label)

print(f"Loaded {len(df)} products")
print(f"Label distribution:\n{df['GT'].value_counts()}\n")

# ===============================
# PRE-COMPUTE CATEGORY EMBEDDINGS
# Do this once outside the loop — much faster
# ===============================
print("Pre-computing category embeddings...")
cat_embeddings = {}
for cat, definition in category_defs.items():
    cat_embeddings[cat] = embed_model.encode(definition, convert_to_tensor=True)
print("Category embeddings ready.\n")

# ===============================
# BERT SUGGESTION
# ===============================
def get_bert_suggestion(product_name, product_description):
    product_text = f"{product_name}: {product_description}"
    prod_emb     = embed_model.encode(product_text, convert_to_tensor=True)

    scores = {}
    for cat, cat_emb in cat_embeddings.items():
        scores[cat] = float(util.cos_sim(prod_emb, cat_emb).item())

    ranked    = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_cat   = ranked[0][0]
    top_score = ranked[0][1]
    sec_score = ranked[1][1] if len(ranked) > 1 else 0.0
    margin    = round(top_score - sec_score, 4)

    return {
        "prediction": top_cat,
        "confidence": round(top_score, 4),
        "margin":     margin,
        "all_scores": {lbl: round(sc, 4) for lbl, sc in ranked},
    }

# ===============================
# FLAN-T5 SUGGESTION
# With robust 3-layer parsing
# ===============================
def get_flan_suggestion(product_name, product_description):
    product_text = f"{product_name}: {str(product_description)[:300]}"

    prompt = f"""You are a product classification expert.

Classify this product into ONE category:
Food, Drinks, Home Care, Personal Care, Pet Food, Other

Product:
{product_text}

Return ONLY:
Category: <name>
Confidence: <0.0 to 1.0>
"""

    inputs  = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    outputs = llm_model.generate(**inputs, max_new_tokens=50)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ── 3-layer parsing ───────────────────────────────────────

    # Layer 1 — direct category name in response
    pred = None
    resp_lower = response.lower()

    # Priority order — check specific before general
    # "Pet Food" before "Food" to avoid wrong match
    priority = ["Pet Food", "Personal Care", "Home Care", "Drinks", "Food", "Other"]
    for cat in priority:
        if cat.lower() in resp_lower:
            pred = cat
            break

    # Layer 2 — regex extract after "Category:"
    if pred is None:
        cat_match = re.search(
            r'(?:category|class|label)\s*[:\-]\s*([^\n\r,\.]+)',
            resp_lower
        )
        if cat_match:
            extracted = cat_match.group(1).strip()
            for cat in priority:
                if cat.lower() in extracted or extracted in cat.lower():
                    pred = cat
                    break

    # Layer 3 — fallback
    if pred is None:
        pred = "Other"

    # Extract confidence
    conf = 0.5
    conf_match = re.search(r'(\d\.\d+)', response)
    if conf_match:
        try:
            conf = float(conf_match.group(1))
            if conf > 1.0:
                conf = conf / 100.0
        except:
            conf = 0.5

    return {
        "prediction": pred,
        "confidence": round(conf, 4),
        "raw":        response[:100],
    }

# ===============================
# MAIN LOOP
# ===============================
print(f"Running pipeline on {len(df)} products...")
print("─" * 75)

bert_preds    = []
bert_confs    = []
bert_margins  = []
flan_preds    = []
flan_confs    = []
flan_raws     = []
same_list     = []
decisions     = []
final_preds   = []

for i, row in df.iterrows():
    product_name = str(row["Category_Clean"])
    product_desc = str(row["category_description"])
    true_label   = row["GT"]

    # BERT
    bert = get_bert_suggestion(product_name, product_desc)

    # FLAN
    flan = get_flan_suggestion(product_name, product_desc)

    # FIX 3 — CONSENSUS CHECK
    same = (bert["prediction"] == flan["prediction"])
    if same:
        final    = bert["prediction"]
        f_conf   = round((bert["confidence"] + flan["confidence"]) / 2, 4)
        decision = "CONSENSUS"
    else:
        final    = "CONFLICT"
        f_conf   = 0.0
        decision = "CONFLICT"

    bert_preds.append(bert["prediction"])
    bert_confs.append(bert["confidence"])
    bert_margins.append(bert["margin"])
    flan_preds.append(flan["prediction"])
    flan_confs.append(flan["confidence"])
    flan_raws.append(flan["raw"])
    same_list.append(same)
    decisions.append(decision)
    final_preds.append(final)

    b_ok   = "✓" if bert["prediction"] == true_label else "✗"
    f_ok   = "✓" if flan["prediction"] == true_label else "✗"
    status = "✅ AGREE" if same else "⚠️  CONFLICT"

    print(f"[{i+1:3d}] {product_name:<35} "
          f"BERT:{bert['prediction']:<15}{b_ok}  "
          f"FLAN:{flan['prediction']:<15}{f_ok}  "
          f"{status}")

# ===============================
# SAVE RESULTS
# ===============================
df["BERT_Pred"]    = bert_preds
df["BERT_Conf"]    = bert_confs
df["BERT_Margin"]  = bert_margins
df["FLAN_Pred"]    = flan_preds
df["FLAN_Conf"]    = flan_confs
df["FLAN_Raw"]     = flan_raws
df["Same"]         = same_list
df["Decision"]     = decisions
df["Final_Answer"] = final_preds

df.to_excel("FINAL_RESULTS_FLAN.xlsx", index=False)

# ===============================
# SUMMARY
# ===============================
total     = len(df)
consensus = (pd.Series(decisions) == "CONSENSUS").sum()
conflict  = (pd.Series(decisions) == "CONFLICT").sum()
bert_acc  = accuracy_score(df["GT"], df["BERT_Pred"])
flan_acc  = accuracy_score(df["GT"], df["FLAN_Pred"])

cons_mask = pd.Series(decisions) == "CONSENSUS"
cons_acc  = accuracy_score(
    df["GT"][cons_mask],
    pd.Series(final_preds)[cons_mask]
) if consensus > 0 else 0.0

print(f"\n{'='*75}")
print("RESULTS SUMMARY — FLAN-T5-Large (FREE · Local · Colab)")
print(f"{'='*75}")
print(f"Total products:          {total}")
print(f"Consensus (both agreed): {consensus} ({consensus/total*100:.1f}%)")
print(f"Conflict (need TB):      {conflict}  ({conflict/total*100:.1f}%)")
print(f"\nBERT accuracy:           {bert_acc*100:.2f}%")
print(f"FLAN accuracy:           {flan_acc*100:.2f}%")
print(f"Consensus accuracy:      {cons_acc*100:.2f}%  ← key metric")

print("\n📊 BERT Report:")
print(classification_report(df["GT"], df["BERT_Pred"], zero_division=0))

print("\n📊 FLAN-T5-Large Report:")
print(classification_report(df["GT"], df["FLAN_Pred"], zero_division=0))

# Show conflicts for analysis
conflict_df = df[pd.Series(decisions) == "CONFLICT"][
    ["Category_Clean", "GT", "BERT_Pred", "FLAN_Pred", "BERT_Conf", "FLAN_Conf"]
]
print(f"\n⚠️  {len(conflict_df)} conflicts need tie-breaker:")
print(conflict_df.to_string(index=False))

print(f"\n✅ DONE — Saved to FINAL_RESULTS_FLAN.xlsx")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FLAN-T5-Large...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Models ready.

Categories: ['Food', 'Drinks', 'Home Care', 'Personal Care', 'Pet Food', 'Other']

Loaded 157 products
Label distribution:
GT
Food             94
Drinks           30
Home Care        18
Personal Care    10
Other             3
Pet Food          2
Name: count, dtype: int64

Pre-computing category embeddings...
Category embeddings ready.

Running pipeline on 157 products...
───────────────────────────────────────────────────────────────────────────
[  1] MINERAL WATER                       BERT:Drinks         ✓  FLAN:Drinks         ✓  ✅ AGREE
[  2] COLAS                               BERT:Drinks         ✓  FLAN:Drinks         ✓  ✅ AGREE
[  3] LIME LEMON SOFT DRINK               BERT:Drinks         ✓  FLAN:Drinks         ✓  ✅ AGREE
[  4] SPARKLING FRUIT SOFT DRINK          BERT:Drinks         ✓  FLAN:Drinks         ✓  ✅ AGREE
[  5] TONIC WATERS                        BERT:Drinks         ✓  FLAN:Drinks         ✓  ✅ AGREE
[  6] SPORTS DRINKS                       BERT:Drinks  